In [ ]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpls8vg8pk".


## Ejercicio 01




In [ ]:
%%cuda
#include "cuda_runtime.h"
#include "device_launch_parameters.h"
#include <stdio.h>
#include <stdlib.h>

__global__ void helloworld() {
    printf("Hola Mundo..! Soy el hilo con id bloque: {%d %d}, id de hilo{ % d % d % d }\n",
        blockIdx.x, blockIdx.y, threadIdx.x, threadIdx.y, threadIdx.z);
}
int main()
{
    helloworld << < 2, 2 >> > (); //--se define un hilo por bloque
    cudaDeviceSynchronize();
    return 0;
}


Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  0  0  0 }
Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  1  0  0 }
Hola Mundo..! Soy el hilo con id bloque: {1 0}, id de hilo{  0  0  0 }
Hola Mundo..! Soy el hilo con id bloque: {1 0}, id de hilo{  1  0  0 }



## Ejercicio 02

In [ ]:
%%cuda
#include "cuda_runtime.h"
#include "device_launch_parameters.h"
#include <stdio.h>
#include <stdlib.h>

__global__ void helloworld()
{
	printf("Hola Mundo..! Soy el hilo con id bloque: {%d %d}, id de hilo{ % d % d % d }\n", blockIdx.x, blockIdx.y, threadIdx.x, threadIdx.y, threadIdx.z);

}
int main()
{
	dim3 threads(1, 2, 4); //--nro de hilos por bloque
	dim3 grid(2, 1); //--nro de bloques por grid
	helloworld << < grid, threads >> > (); //--se define un hilo por bloque
  cudaDeviceSynchronize();
	return 0;
}


Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  0  0  0 }
Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  0  1  0 }
Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  0  0  1 }
Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  0  1  1 }
Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  0  0  2 }
Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  0  1  2 }
Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  0  0  3 }
Hola Mundo..! Soy el hilo con id bloque: {0 0}, id de hilo{  0  1  3 }
Hola Mundo..! Soy el hilo con id bloque: {1 0}, id de hilo{  0  0  0 }
Hola Mundo..! Soy el hilo con id bloque: {1 0}, id de hilo{  0  1  0 }
Hola Mundo..! Soy el hilo con id bloque: {1 0}, id de hilo{  0  0  1 }
Hola Mundo..! Soy el hilo con id bloque: {1 0}, id de hilo{  0  1  1 }
Hola Mundo..! Soy el hilo con id bloque: {1 0}, id de hilo{  0  0  2 }
Hola Mundo..! Soy el hilo con id bloque: {1 0}, id de hilo{  0  1  2 }
Hola M

## Ejercicio 03

In [ ]:
%%cuda
#include "cuda_runtime.h"
#include "device_launch_parameters.h"
#include <stdio.h>
#include <stdlib.h>
// declaracion de funciones
// GLOBAL: funcion llamada desde el host y ejecutada en el device (kernel)

__global__ void suma_GPU(int a, int b, int* c)
{
  *c = a + b;
}
// HOST: funcion llamada y ejecutada desde el host
__host__ int suma_CPU(int a, int b)
{
   return (a + b);
}
int main()
{
  // declaraciones
  int n1 = 1, n2 = 2, c = 0;
  int* hst_c;
  int m1 = 10, m2 = 20;
  int* dev_c;
  // reserva en el host
  hst_c = (int*)malloc(sizeof(int));
  // reserva en el device
  cudaMalloc((void**)&dev_c, sizeof(int));
  // llamada a la funcion suma_CPU
  c = suma_CPU(n1, n2);
  // resultados CPU
  printf("CPU:\n");
  printf("%2d + %2d = %2d \n", n1, n2, c);
  // llamada a la funcion suma_GPU
  suma_GPU << <1, 1 >> > (m1, m2, dev_c);
  // recogida de datos desde el device
  cudaMemcpy(hst_c, dev_c, sizeof(int), cudaMemcpyDeviceToHost);
  // resultados GPU
  printf("GPU:\n");
  printf("%2d + %2d = %2d \n", m1, m2, *hst_c);
  // salida
  printf("\npulsa INTRO para finalizar...");
  fflush(stdin);
  char tecla = getchar();
  return 0;
}

CPU:
 1 +  2 =  3 
GPU:
10 + 20 = 30 

pulsa INTRO para finalizar...


## Ejercicio 04

In [ ]:
%%cuda
#include <stdio.h>
#include <stdlib.h>
#include "cuda_runtime.h"
#include "device_launch_parameters.h"
#define N 16 // tamaño de los vectores
#define BLOCK 5 // tamaño del bloque
// declaracion de funciones
// GLOBAL: función llamada desde el host y ejecutada en el device (kernel)

__global__ void suma(float* a, float* b, float* c)
{
    int myID = threadIdx.x + blockDim.x * blockIdx.x;
    // Sólo trabajan N hilos
   if (myID < N)
   {
      c[myID] = a[myID] + b[myID];
   }
}
// MAIN: rutina principal ejecutada en el host
int main(int argc, char** argv)
{
  // declaraciones
  float* vector1, * vector2, * resultado;
  float* dev_vector1, * dev_vector2, * dev_resultado;
  // reserva memoria en el host
  vector1 = (float*)malloc(N * sizeof(float));
  vector2 = (float*)malloc(N * sizeof(float));
  resultado = (float*)malloc(N * sizeof(float));
  // reserva memoria en el device
  cudaMalloc((void**)&dev_vector1, N * sizeof(float));
  cudaMalloc((void**)&dev_vector2, N * sizeof(float));
  cudaMalloc((void**)&dev_resultado, N * sizeof(float));
  // inicialización de vectores con valores aleatorios
  for (int i = 0; i < N; i++)
  {
    vector1[i] = (float)rand() / RAND_MAX;
    vector2[i] = (float)rand() / RAND_MAX;
  }
  // copia de datos hacia el device
  cudaMemcpy(dev_vector1, vector1, N * sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(dev_vector2, vector2, N * sizeof(float), cudaMemcpyHostToDevice);
  // lanzamiento del kernel
  // calculamos el número de bloques necesario para un tamaño de bloque fijo
  int nBloques = N / BLOCK;
  if (N % BLOCK != 0)
  {
   nBloques = nBloques + 1;
  }
  int hilosB = BLOCK;
  printf("Vector de %d elementos\n", N);
  printf("Lanzamiento con %d bloques (%d hilos)\n", nBloques, nBloques * hilosB);
  suma << < nBloques, hilosB >> > (dev_vector1, dev_vector2, dev_resultado);
  // recogida de datos desde el device
  cudaMemcpy(resultado, dev_resultado, N * sizeof(float), cudaMemcpyDeviceToHost);

  // impresión de resultados en el host
  printf("> vector1:\n");
  for (int i = 0; i < N; i++)
  {
    printf("%.2f ", vector1[i]);
  }
  printf("\n");
  printf("> vector2:\n");
  for (int i = 0; i < N; i++)
  {
    printf("%.2f ", vector2[i]);
  }
  printf("\n");
  printf("> SUMA:\n");
  for (int i = 0; i < N; i++)
  {
    printf("%.2f ", resultado[i]);
  }
  printf("\n");
  // liberamos memoria en el device
  cudaFree(dev_vector1);
  cudaFree(dev_vector2);
  cudaFree(dev_resultado);
  // salida
   printf("\npulsa INTRO para finalizar...");
  fflush(stdin);
  char tecla = getchar();
  return 0;
}

Vector de 16 elementos
Lanzamiento con 4 bloques (20 hilos)
> vector1:
0.84 0.78 0.91 0.34 0.28 0.48 0.36 0.95 0.64 0.14 0.02 0.14 0.16 0.13 1.00 0.51 
> vector2:
0.39 0.80 0.20 0.77 0.55 0.63 0.51 0.92 0.72 0.61 0.24 0.80 0.40 0.11 0.22 0.84 
> SUMA:
1.23 1.58 1.11 1.10 0.83 1.11 0.88 1.87 1.35 0.75 0.26 0.94 0.56 0.24 1.22 1.35 

pulsa INTRO para finalizar...


## Ejercicio 05

In [ ]:
%%cuda
#include "cuda_runtime.h"
#include "device_launch_parameters.h"
#include <iostream>
#include <stdio.h>
using namespace std;
//-- Funcion device recursiva
__device__ int Factorial(int x) {
  if (x == 0)
    return 1;
  else
    return x * Factorial(x - 1);

}
//--kernel
__global__ void MyKernel(int n, int * fact) {
   *fact = Factorial(n);
}
int main() {
  int n, resultado, * d_f;
  //--Leer un entero
  //--cout << "Ingrese un número: ";
  //--cin >> n;
  n = 8;
  //--reservar memoria en el device
  cudaMalloc(&d_f, sizeof(int));
  //--Lanzar el kernel
  MyKernel << <1, 1 >> > (n, d_f); //--1 bloque y un hilo por bloque
  //--Copiar el resultado al host
  cudaMemcpy(&resultado, d_f, sizeof(int), cudaMemcpyDeviceToHost);
  // Print out
  cout << "El resultado es: " << resultado << endl;
  cudaFree(d_f); //--libera la memoria del host
  system("pause");
  return 0;
}

El resultado es: 40320
sh: 1: pause: not found



In [ ]:
%%cuda
#include "cuda_runtime.h"

/usr/bin/ld: /usr/lib/gcc/x86_64-linux-gnu/11/../../../x86_64-linux-gnu/Scrt1.o: in function `_start':
(.text+0x1b): undefined reference to `main'
collect2: error: ld returned 1 exit status



In [ ]:
%%cuda
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define DX 4

#define DY 4

#define DZ 4

__constant__ float escala_GPU[DX];   // factor por columna


__global__ void procesar3D(float *d_mat, float *d_res) {

    __shared__ float tile[DY][DX];   // tile del bloque

    // Coordenadas globales del thread

    int gx = blockIdx.x*blockDim.x + threadIdx.x;

    int gy = blockIdx.y*blockDim.y + threadIdx.y;

    int gz = blockIdx.z*blockDim.z + threadIdx.z;

    if(gx>=DX || gy>=DY || gz>=DZ) return;

    // Índice lineal en memoria global

    int idx = gz*(DY*DX) + gy*DX + gx;

    // ── Cargar en shared memory ──
    tile[threadIdx.y][threadIdx.x] = d_mat[idx];

    __syncthreads();   // barrera de sincronización

    // ── Usar memoria constante ──
    float factor = escala_GPU[gx];   // cache automático

    // ── Procesar desde shared memory ──
    float val = tile[threadIdx.y][threadIdx.x];

    d_res[idx] = val * factor + gz;

}

// ── MAIN: reserva host y device ──
int main() {

    int total = DX*DY*DZ;   // 64 elementos

    // Reserva HOST (CPU)
    float *h_mat    = (float*)malloc(total*sizeof(float));

    float *h_res    = (float*)malloc(total*sizeof(float));

    float  h_escala[DX] = {1.0f, 2.0f, 3.0f, 4.0f};

    // Inicializar datos
    for(int i=0;i<total;i++) h_mat[i]=(float)(i+1);

    // Reserva DEVICE (GPU)
    float *d_mat, *d_res;

    cudaMalloc((void**)&d_mat, total*sizeof(float));

    cudaMalloc((void**)&d_res, total*sizeof(float));

    // Copiar a memoria constante
    cudaMemcpyToSymbol(escala_GPU, h_escala, DX*sizeof(float));

    // Copiar datos Host → Device
    cudaMemcpy(d_mat,h_mat,total*sizeof(float),cudaMemcpyHostToDevice);

    // Configuración de bloques 3D (2×2×2 = 8 threads/bloque)
    dim3 block(2, 2, 2);

    dim3 grid(DX/2, DY/2, DZ/2);   // 2×2×2 = 8 bloques

    procesar3D<<<grid, block>>>(d_mat, d_res);

    // Copiar resultado Device → Host
    cudaMemcpy(h_res,d_res,total*sizeof(float),cudaMemcpyDeviceToHost);
    // Imprimir resultado
    for(int z=0;z<DZ;z++)

      for(int y=0;y<DY;y++)

        for(int x=0;x<DX;x++){

          int i=z*DY*DX+y*DX+x;

          printf("[%d][%d][%d]=%6.1f\n",z,y,x,h_res[i]);

        }

    // Liberar memoria
    cudaFree(d_mat); cudaFree(d_res);

    free(h_mat);    free(h_res);

    return 0;

}

[0][0][0]=   1.0
[0][0][1]=   4.0
[0][0][2]=   9.0
[0][0][3]=  16.0
[0][1][0]=   5.0
[0][1][1]=  12.0
[0][1][2]=  21.0
[0][1][3]=  32.0
[0][2][0]=   9.0
[0][2][1]=  20.0
[0][2][2]=  33.0
[0][2][3]=  48.0
[0][3][0]=  13.0
[0][3][1]=  28.0
[0][3][2]=  45.0
[0][3][3]=  64.0
[1][0][0]=   2.0
[1][0][1]=   5.0
[1][0][2]=  10.0
[1][0][3]=  17.0
[1][1][0]=   6.0
[1][1][1]=  13.0
[1][1][2]=  22.0
[1][1][3]=  33.0
[1][2][0]=  10.0
[1][2][1]=  21.0
[1][2][2]=  34.0
[1][2][3]=  49.0
[1][3][0]=  14.0
[1][3][1]=  29.0
[1][3][2]=  46.0
[1][3][3]=  65.0
[2][0][0]=  35.0
[2][0][1]=  70.0
[2][0][2]= 107.0
[2][0][3]= 146.0
[2][1][0]=  39.0
[2][1][1]=  78.0
[2][1][2]= 119.0
[2][1][3]= 162.0
[2][2][0]=  43.0
[2][2][1]=  86.0
[2][2][2]= 131.0
[2][2][3]= 178.0
[2][3][0]=  47.0
[2][3][1]=  94.0
[2][3][2]= 143.0
[2][3][3]= 194.0
[3][0][0]=  36.0
[3][0][1]=  71.0
[3][0][2]= 108.0
[3][0][3]= 147.0
[3][1][0]=  40.0
[3][1][1]=  79.0
[3][1][2]= 120.0
[3][1][3]= 163.0
[3][2][0]=  44.0
[3][2][1]=  87.0
[3][2][2]= 132